In [1]:
# Imports and project configuration

from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

import os
import pandas as pd
import plotly.express as px

In [2]:
# Project paths

PROJECT_DIR = Path.cwd().parent
OUTPUT_DIR = PROJECT_DIR / "data" / "outputs"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Load environment variables

load_dotenv(PROJECT_DIR / ".env")

RDS_HOST = os.getenv("RDS_HOST")
RDS_PORT = os.getenv("RDS_PORT")
RDS_DB = os.getenv("RDS_DB")
RDS_USER = os.getenv("RDS_USER")
RDS_PASSWORD = os.getenv("RDS_PASSWORD")

In [3]:
# Connect to PostgreSQL RDS

password_encoded = quote_plus(RDS_PASSWORD)

DATABASE_URL = (
    f"postgresql+psycopg2://{RDS_USER}:{password_encoded}"
    f"@{RDS_HOST}:{RDS_PORT}/{RDS_DB}"
)

engine = create_engine(DATABASE_URL)

# Test it

with engine.connect() as connection:
    result = connection.execute(
        text("SELECT CURRENT_DATABASE();")
    )

    print("Connected to:", result.fetchone()[0])

Connected to: kayak_db


In [4]:
# Load Top 5 destinations from RDS
top5_df = pd.read_sql(
    """
    SELECT *
    FROM top_5_destinations
    ORDER BY weather_score DESC;
    """,
    engine
)

top5_df

,city_id,city,latitude,longitude,avg_temp_day,avg_temp_min,avg_temp_max,avg_humidity,avg_wind_speed,avg_pop,total_rain,temperature_score,rain_penalty,pop_penalty,wind_penalty,humidity_penalty,weather_score
0,5,Rouen,49.440459,1.093966,24.384286,15.421429,27.242857,45.428571,6.204286,0.197143,8.06,98.078571,16.12,19.714286,31.021429,21.857143,27.063571
1,6,Paris,48.853495,2.348391,25.417143,19.762857,29.638571,41.857143,4.685714,0.234286,5.19,92.914286,10.38,23.428571,23.428571,27.214286,26.181429
2,35,La Rochelle,46.159732,-1.151595,22.961429,18.761429,23.445714,62.714286,6.921429,0.291429,6.71,94.807143,13.42,29.142857,34.607143,4.071429,25.075000
3,7,Amiens,49.894171,2.295695,24.185714,15.728571,27.224286,44.428571,6.568571,0.271429,9.92,99.071429,19.84,27.142857,32.842857,23.357143,24.787857
4,1,Mont Saint Michel,48.635954,-1.511460,22.175714,16.080000,23.435714,60.428571,6.042857,0.171429,15.06,90.878571,30.12,17.142857,30.214286,0.642857,22.339286


In [5]:
top5_df.shape

(5, 17)

In [6]:
# Inspect the Top 5

top5_df[
    [
        "city",
        "avg_temp_day",
        "total_rain",
        "avg_pop",
        "weather_score"
    ]
]

,city,avg_temp_day,total_rain,avg_pop,weather_score
0,Rouen,24.384286,8.06,0.197143,27.063571
1,Paris,25.417143,5.19,0.234286,26.181429
2,La Rochelle,22.961429,6.71,0.291429,25.075000
3,Amiens,24.185714,9.92,0.271429,24.787857
4,Mont Saint Michel,22.175714,15.06,0.171429,22.339286


In [7]:
# Round them for readability

top5_df["avg_temp_day"] = (
    top5_df["avg_temp_day"].round(1)
)

top5_df["total_rain"] = (
    top5_df["total_rain"].round(1)
)

top5_df["avg_pop"] = (
    (top5_df["avg_pop"] * 100).round(1)
)

top5_df["weather_score"] = (
    top5_df["weather_score"].round(1)
)

In [8]:
# Create the Top 5 destinations map

fig_destinations = px.scatter_map(
    top5_df,
    lat="latitude",
    lon="longitude",
    size="weather_score",
    hover_name="city",
    hover_data={
        "avg_temp_day": True,
        "total_rain": True,
        "avg_pop": True,
        "weather_score": True,
        "latitude": False,
        "longitude": False
    },
    zoom=4.5,
    height=650,
    title="Top 5 Destinations in France Based on 7-Day Weather Forecast"
)

fig_destinations.update_layout(
    map_style="open-street-map",
    margin={
        "r": 0,
        "t": 60,
        "l": 0,
        "b": 0
    }
)

fig_destinations.show()

In [9]:
# Save the destinations map
destinations_map_path = (
    OUTPUT_DIR
    / "top_5_destinations.html"
)

fig_destinations.write_html(
    destinations_map_path
)

print(
    "Saved:",
    destinations_map_path
)

Saved: c:\Users\Alex\Desktop\Jehda AI\Fullstack\CDSD Certification\1_Kayak\data\outputs\top_5_destinations.html


In [10]:
# Load hotels from RDS
hotels_df = pd.read_sql(
    """
    SELECT *
    FROM hotels;
    """,
    engine
)

hotels_df.head()

,hotel_id,city_id,name,rating,description,url,city,latitude,longitude
0,1,5,"Le Clémenceau - 6 pers, balcon & cœur de Rouen",NaN,Spacious Accommodations: Le Clémenceau in Roue...,https://www.booking.com/hotel/fr/le-clemenceau...,Rouen,49.434768,1.088898
1,2,5,ibis budget Rouen Centre Rive Gauche,8.3,Hotel ibis budget Rouen Center Rive Gauche Gau...,https://www.booking.com/hotel/fr/ibis-budget-r...,Rouen,49.425341,1.068718
2,3,5,"Radisson Blu Hotel, Rouen Centre",8.9,Comfortable Accommodations: Rooms feature air ...,https://www.booking.com/hotel/fr/radisson-blu-...,Rouen,49.446441,1.094120
3,4,5,Hyatt Place Rouen,8.6,Comfortable Accommodations: Hyatt Place Rouen ...,https://www.booking.com/hotel/fr/hyatt-place-r...,Rouen,49.453286,1.098285
4,5,5,B&B HOTEL Rouen Centre,6.8,"Ideally located for exploring Rouen, a city of...",https://www.booking.com/hotel/fr/comforthotelr...,Rouen,49.430326,1.084833


In [11]:
# Data-quality check
hotels_df.isna().sum()

hotel_id        0
city_id         0
name            0
rating         12
description     0
url             0
city            0
latitude        0
longitude       0
dtype: int64

In [12]:
# Check coordinates are numeric:
hotels_df["latitude"] = pd.to_numeric(
    hotels_df["latitude"],
    errors="coerce"
)

hotels_df["longitude"] = pd.to_numeric(
    hotels_df["longitude"],
    errors="coerce"
)

hotels_df["rating"] = pd.to_numeric(
    hotels_df["rating"],
    errors="coerce"
)

In [13]:
#Select Top 20 hotels
top20_hotels = (
    hotels_df
    .dropna(
        subset=[
            "rating",
            "latitude",
            "longitude"
        ]
    )
    .sort_values(
        "rating",
        ascending=False
    )
    .head(20)
    .copy()
)


top20_hotels[
    [
        "name",
        "city",
        "rating"
    ]
]

,name,city,rating
79,l'esprit d'EMMA,Amiens,9.8
93,Appart Standing - La Coque d'Or - Mont-St-Michel,Mont Saint Michel,9.6
78,Elegant 120m2-Face cathedrale-6 pers-free Parking,Amiens,9.5
13,Le petit Rouen,Rouen,9.5
99,Résidence Beauvoir le Mont-Saint-Michel,Mont Saint Michel,9.4
15,Joli appart cosy tout confort avec parking gra...,Rouen,9.4
10,Duplex vieux marché quartier historique wifi/p...,Rouen,9.4
11,Le Majolique Nice and Cozy Apartment,Rouen,9.4
62,Ginkgo Maison d'hôtes,Amiens,9.3
19,Le Petit Horloge,Rouen,9.3


In [14]:
# Check representation by city

top20_hotels["city"].value_counts()

city
Amiens               6
Rouen                6
Mont Saint Michel    3
Paris                3
La Rochelle          2
Name: count, dtype: int64

In [15]:
# Create the Top 20 hotels map

fig_hotels = px.scatter_map(
    top20_hotels,
    lat="latitude",
    lon="longitude",
    size="rating",
    hover_name="name",
    hover_data={
        "city": True,
        "rating": True,
        "latitude": False,
        "longitude": False
    },
    zoom=4.5,
    height=700,
    title="Top 20 Hotels in the Best Weather Destinations"
)

fig_hotels.update_layout(
    map_style="open-street-map",
    margin={
        "r": 0,
        "t": 60,
        "l": 0,
        "b": 0
    }
)

fig_hotels.show()

In [16]:
# Check

top20_hotels.groupby("city").agg(
    hotels=("name", "count"),
    min_lat=("latitude", "min"),
    max_lat=("latitude", "max"),
    min_lon=("longitude", "min"),
    max_lon=("longitude", "max")
)

,hotels,min_lat,max_lat,min_lon,max_lon
city,,,,,
Amiens,6,49.879141,49.899693,2.264739,2.313307
La Rochelle,2,46.168208,46.169697,-1.161885,-1.138887
Mont Saint Michel,3,48.597878,48.636166,-1.510476,-1.508419
Paris,3,48.844958,48.882112,2.296698,2.335513
Rouen,6,49.431371,49.450848,1.084756,1.104508


In [17]:
# City zoom
test_city = top20_hotels["city"].value_counts().index[0]

city_hotels = top20_hotels[
    top20_hotels["city"] == test_city
].copy()

print(test_city)
print("Hotels:", len(city_hotels))

fig_test = px.scatter_map(
    city_hotels,
    lat="latitude",
    lon="longitude",
    hover_name="name",
    hover_data={
        "city": True,
        "rating": True,
        "latitude": True,
        "longitude": True
    },
    zoom=12,
    height=650,
    title=f"Hotels in {test_city}"
)

fig_test.update_traces(
    marker={"size": 14}
)

fig_test.update_layout(
    map_style="open-street-map"
)

fig_test.show()

Amiens
Hotels: 6


In [18]:
# Create Top 20 hotels map with a tighter automatic view

import plotly.express as px

# Keep only rows with valid coordinates
top20_hotels_map = top20_hotels.dropna(
    subset=["latitude", "longitude"]
).copy()

# Calculate geographic center of all hotels
center_lat = top20_hotels_map["latitude"].mean()
center_lon = top20_hotels_map["longitude"].mean()

# Create map
fig_hotels = px.scatter_map(
    top20_hotels_map,
    lat="latitude",
    lon="longitude",
    hover_name="name",
    hover_data={
        "city": True,
        "rating": True,
        "latitude": False,
        "longitude": False
    },
    center={
        "lat": center_lat,
        "lon": center_lon
    },
    zoom=6.3,
    height=750,
    title="Top 20 Hotels in the Best Weather Destinations"
)

# Improve marker visibility
fig_hotels.update_traces(
    marker={
        "size": 11,
        "opacity": 0.75
    }
)

# Clean layout
fig_hotels.update_layout(
    map_style="open-street-map",
    margin={
        "r": 0,
        "t": 60,
        "l": 0,
        "b": 0
    }
)

fig_hotels.show()

In [19]:
# Save the hotel map

hotels_map_path = (
    OUTPUT_DIR
    / "top_20_hotels.html"
)

fig_hotels.write_html(
    hotels_map_path
)

print(
    "Saved:",
    hotels_map_path
)

Saved: c:\Users\Alex\Desktop\Jehda AI\Fullstack\CDSD Certification\1_Kayak\data\outputs\top_20_hotels.html


In [20]:
# Save the Top 20 dataset

top20_hotels.to_csv(
    OUTPUT_DIR / "top_20_hotels.csv",
    index=False
)

In [21]:
# Print
print("Top 5 destinations:", len(top5_df))
print("Top 20 hotels:", len(top20_hotels))

print("\nTop destinations:")
print(top5_df["city"].tolist())

print("\nHotel distribution:")
print(top20_hotels["city"].value_counts())

Top 5 destinations: 5
Top 20 hotels: 20

Top destinations:
['Rouen', 'Paris', 'La Rochelle', 'Amiens', 'Mont Saint Michel']

Hotel distribution:
city
Amiens               6
Rouen                6
Mont Saint Michel    3
Paris                3
La Rochelle          2
Name: count, dtype: int64


## Conclusion

The visualization layer uses data stored in the PostgreSQL data warehouse hosted on AWS RDS.

The first map presents the five French destinations with the highest weather scores based on the seven-day forecast.

The second map presents the twenty highest-rated hotels identified within the selected destinations.

These visualizations complete the end-to-end data pipeline:

**API and web data collection → transformation → S3 data lake → RDS data warehouse → analytical visualization.**

In [22]:
# Export static previews for GitHub README
fig_destinations.write_image(
    OUTPUT_DIR / "top_5_destinations.png",
    width=1200,
    height=700,
    scale=2
)

fig_hotels.write_image(
    OUTPUT_DIR / "top_20_hotels.png",
    width=1200,
    height=700,
    scale=2
)

print("PNG maps saved successfully.")

PNG maps saved successfully.
